Restructure data extracted from an almagamated .csv file

In [2]:
import pandas as pd

test = pd.read_csv('./aggregate/test.csv')
train = pd.read_csv('./aggregate/train.csv')
val = pd.read_csv('./aggregate/valid.csv')

tef = test.loc[(test.Fear == 1 )]
vaf = val.loc[(val.Fear == 1 )]

f_cat = pd.concat([tef, vaf], ignore_index=True)

new_fear = pd.concat([train, f_cat], ignore_index=True)

new_fear.iloc[:, 2:11].sum()

new_fear.to_csv('fear_added_train.csv')

In [3]:
print(
    test.loc[
        (test['text'].str.contains('link', case=False, regex=False, na=False)) & 
        (test['Dataset'] == 'SemEval'), 
        ['text']
    ].to_string()
)

                                                                                                                                         text
5552             [NAME] [NAME] has occupied one full lane for toll collection permanently resulting in traffic snarl everyday on UP Link Road
6536  The gravediggers looked solemn. The groundhog blinked back tears. The crows fell silent. Hey, what about ME! crowed the rooster. vss365
6624                                               [NAME] [NAME] And just saw your LinkedIn comment after I sent this! Thanks for the message
6679                               just released two videos in honer of Cory Monteith and Finn Hundson go wath link in bio [NAME] [NAME] glee
6839       A [NAME] egofest AGAIN on LinkedIn. Advocate for a change. arrogance is NOT advocacy. So tired of their self promotion wo purpose.
8433                                                                              Is it national dont use your blinker day or something? Wtf!
8677  

In [135]:
new_fear.iloc[:, 2:11].columns

Index(['Joy', 'Trust', 'Fear', 'Surprise', 'Sadness', 'Disgust', 'Anger',
       'Anticipation', 'Neutral'],
      dtype='str')

In [ ]:
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

label_cols = ['Joy', 'Trust', 'Fear', 'Surprise', 'Sadness', 'Disgust', 'Anger', 'Anticipation', 'Neutral']

df=new_fear

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Generate train/test indices based on the multilabel matrix
for train_idx, test_idx in msss.split(df, df[label_cols].values):
    tr_fa = df.iloc[train_idx].reset_index(drop=True)
    te_fa = df.iloc[test_idx].reset_index(drop=True)
    
balance_check = pd.DataFrame({
    'Train %': tr_fa[label_cols].mean() * 100,
    'Test %': te_fa[label_cols].mean() * 100,
    'OG %': train[label_cols].mean() * 100
})

tr_fa.to_csv('tr_fa.csv')
te_fa.to_csv('te_fa.csv')

tr_len = [len(tr_fa.loc[tr_fa['Dataset'] == 'GoEmotions']),
len(tr_fa.loc[tr_fa['Dataset'] == 'SemEval'])]


te_len = [len(te_fa.loc[te_fa['Dataset'] == 'GoEmotions']),
len(te_fa.loc[te_fa['Dataset'] == 'SemEval'])]

# verify the distribution of dataset
print(tr_len[1] / tr_len[0])
print(te_len[1] / te_len[0])


0.17094041554576156
0.16824888482214342
